# 02 — Model Experiments

Compare the three candidate classifiers, inspect the errors that matter for
security, and check what the anomaly detector adds.

**Protocol.** Models are compared on a **validation** split. The test split is
opened exactly once, at the end, for the winner only. Tuning or selecting on
the test set would make the reported test score optimistically biased — the
test labels would have influenced a decision, so the set is no longer held out.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import PATHS, TRAINING, BENIGN_LABEL
from src.data_loader import load_sample_data, normalize_columns
from src.preprocessing import clean_dataset, select_feature_columns, build_preprocessor, balance_classes
from src.train import build_candidate_models, split_data
from src.evaluate import evaluate_model, comparison_table

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

In [ ]:
processed = PATHS.processed_dataset
if processed.exists():
    df = pd.read_parquet(processed); SOURCE = "processed CIC-IDS2017"
elif processed.with_suffix(".csv").exists():
    df = pd.read_csv(processed.with_suffix(".csv")); SOURCE = "processed CIC-IDS2017 (csv)"
else:
    df, _ = clean_dataset(normalize_columns(load_sample_data()), min_samples_per_class=10)
    SOURCE = "SYNTHETIC sample"

print(f"Source: {SOURCE}   shape: {df.shape}")
if "SYNTHETIC" in SOURCE:
    print("\n*** Synthetic data: metrics below validate the PIPELINE, not real-world detection. ***")

## 1. Splits and preprocessing

Three-way stratified split. Stratification matters here because the rarest
class may have only a few dozen rows — an unstratified split can leave it
absent from a fold entirely.

The preprocessor is fitted on **training rows only**. Fitting on the full
dataset first would leak the held-out distribution (the scaler's median and
IQR, the encoder's vocabulary) into training.

In [ ]:
train_df, val_df, test_df = split_data(df, label_column="label")
train_df = balance_classes(train_df, label_column="label")

features = select_feature_columns(train_df.drop(columns=["label"]), restrict_to_core=True)
print(f"train/val/test rows : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"raw features        : {len(features)}")

preprocessor = build_preprocessor(features)
X_train = preprocessor.fit_transform(train_df[features])   # fit on TRAIN only
X_val   = preprocessor.transform(val_df[features])
X_test  = preprocessor.transform(test_df[features])
print(f"encoded feature dim : {X_train.shape[1]}")

encoder = LabelEncoder().fit(sorted(df["label"].unique()))
y_train = encoder.transform(train_df["label"])
class_names = list(encoder.classes_)
sample_weight = compute_sample_weight("balanced", y_train)

## 2. Train the three candidates

- **Logistic Regression** — linear baseline. If a complex model cannot beat it,
  the complexity is not earning its keep.
- **Random Forest** — bagged trees. Captures the conjunctive threshold logic
  that defines attacks ("high packet rate *and* high asymmetry").
- **XGBoost** — gradient boosting; the standard strong baseline on tabular data.

In [ ]:
import time

results, fitted, timings = [], {}, {}
for name, model in build_candidate_models(n_classes=len(class_names)).items():
    start = time.perf_counter()
    if name == "XGBoost":
        model.fit(X_train, y_train, sample_weight=sample_weight)
    else:
        model.fit(X_train, y_train)
    timings[name] = time.perf_counter() - start

    y_pred = encoder.inverse_transform(model.predict(X_val))
    y_proba = model.predict_proba(X_val) if hasattr(model, "predict_proba") else None
    result = evaluate_model(name, val_df["label"], y_pred, y_proba, class_names, split="validation")
    results.append(result); fitted[name] = model
    print(f"{result.summary_line()}   ({timings[name]:.1f}s)")

In [ ]:
table = comparison_table(results)
display(table)

ax = table.set_index("Model")[["Accuracy", "Macro F1", "Macro Recall"]].plot(kind="bar", figsize=(10, 5))
ax.set_ylim(0, 1.05); ax.set_ylabel("Score"); ax.set_title("Validation performance")
ax.legend(loc="lower right"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 3. Why recall, not accuracy

A **false negative** is an attack the system labelled benign — an intrusion
that proceeds undetected. A **false positive** is benign traffic flagged for
review — an analyst spends a few minutes and dismisses it.

These costs are wildly asymmetric, so the model is selected on macro-F1 and
scrutinised on per-class recall. That said, false positives are not free: an
IDS with a high false-alarm rate gets muted by the team that operates it, which
converts every future true positive into a missed one. The goal is high recall
at a false-alarm rate a human queue can absorb.

In [ ]:
best = max(results, key=lambda r: r.macro_f1)
print(f"Selected on validation macro-F1: {best.model_name} ({best.macro_f1:.4f})\n")

per_class = pd.DataFrame(best.per_class).T
per_class = per_class[per_class.index.isin(class_names)]
display(per_class[["precision", "recall", "f1", "support"]].round(3))

ax = per_class["recall"].sort_values().plot(kind="barh", color="indianred", figsize=(9, 4))
ax.set_xlabel("Recall"); ax.set_xlim(0, 1.05)
ax.set_title("Per-class recall — low bars are attack types being missed")
plt.tight_layout(); plt.show()

## 4. Final evaluation on the held-out test split

Opened once, for the winner only.

In [ ]:
model = fitted[best.model_name]
y_test_pred = encoder.inverse_transform(model.predict(X_test))
y_test_proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None
test_result = evaluate_model(best.model_name, test_df["label"], y_test_pred,
                             y_test_proba, class_names, split="test")
print(test_result.summary_line())
print(f"\nFalse negatives (missed attacks) : {test_result.false_negatives:,}")
print(f"False positives (analyst noise)  : {test_result.false_positives:,}")

In [ ]:
matrix = np.array(test_result.confusion_matrix)
normalised = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=test_result.labels, yticklabels=test_result.labels)
axes[0].set_title("Confusion matrix (counts)")
sns.heatmap(normalised, annot=True, fmt=".2f", cmap="Blues", ax=axes[1],
            xticklabels=test_result.labels, yticklabels=test_result.labels)
axes[1].set_title("Row-normalised (recall per class)")
for ax in axes:
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

## 5. Feature importance and explainability

Global importance answers "what does this model rely on overall". SHAP answers
"why *this* flow". Both are useful; only the second is defensible when an
analyst challenges a specific alert.

Read these with the correlation findings from notebook 01 in mind: when two
features are near-duplicates, importance and attribution split between them,
which can make an individually important feature look weak.

In [ ]:
from src.explainability import ModelExplainer
from src.train import ModelBundle
from datetime import datetime, timezone

bundle = ModelBundle(
    preprocessor=preprocessor, model=model, label_encoder=encoder,
    feature_columns=features, class_names=class_names,
    model_name=best.model_name, model_version="notebook",
    trained_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
    training_rows=len(train_df),
)

explainer = ModelExplainer(bundle)
print(f"Explanation method: {explainer.method}")
importance = explainer.global_importance(top_k=20)
display(importance)

ax = importance.set_index("feature")["importance"].sort_values().plot(
    kind="barh", figsize=(9, 7), color="steelblue")
ax.set_title(f"Global feature importance ({explainer.method})"); ax.set_xlabel("Mean |contribution|")
plt.tight_layout(); plt.show()

## 6. What the anomaly detector adds

The Isolation Forest is trained on **benign traffic only**, so it never sees an
attack label. Its job is the case the supervised model structurally cannot
handle: a flow whose attack type was absent from training. The supervised model
must map such a flow onto one of its known classes; the detector can simply say
"this does not look like anything normal".

In [ ]:
from src.anomaly_detection import train_anomaly_detector, score_anomalies

detector = train_anomaly_detector(
    frame=train_df, preprocessor=preprocessor,
    feature_columns=features, label_column="label",
)
scores, flags = score_anomalies(test_df[features], detector)

comparison = pd.DataFrame({
    "label": test_df["label"].values,
    "anomaly_score": scores,
    "flagged": flags,
})
summary = comparison.groupby("label").agg(
    mean_anomaly_score=("anomaly_score", "mean"),
    flagged_rate=("flagged", "mean"),
    n=("flagged", "size"),
).round(3)
display(summary)

ax = sns.boxplot(data=comparison, x="label", y="anomaly_score", showfliers=False)
ax.set_title("Anomaly score by true class (higher = more unusual)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

### Interpreting this

If attack classes show higher mean anomaly scores than BENIGN, the detector is
adding an independent signal. If they do not, it is contributing little and the
composite risk score is effectively driven by the supervised model — worth
knowing and worth saying out loud rather than presenting the ensemble as
automatically better.

The anomaly score is **not a probability**. It is a min-max normalised
Isolation Forest path-length score, calibrated against the training
distribution. A score of 0.94 means "in the top few percent of unusual relative
to training traffic", not "94% likely to be an attack".